In [5]:
import os

import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.manifold import TSNE



In [55]:
combined_df = pd.read_csv('../Data/FinalDataset/merged_all.csv')

In [65]:
combined_df.dropna(subset = 'latitude',inplace=True)
combined_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5364 entries, 0 to 5366
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   id           5364 non-null   int64  
 1   name         5364 non-null   object 
 2   description  5364 non-null   object 
 3   address      5002 non-null   object 
 4   latitude     5364 non-null   float64
 5   longitude    5364 non-null   float64
 6   location     5326 non-null   object 
 7   types        5364 non-null   object 
dtypes: float64(2), int64(1), object(5)
memory usage: 377.2+ KB


In [66]:

# One-hot encode 'Category'
encoder = OneHotEncoder()
category_encoded = encoder.fit_transform(combined_df[['types']]).toarray()

In [67]:
# Combine Latitude, Longitude with encoded features and TF-IDF vectors
geo_features = combined_df[['latitude', 'longitude']].values
geo_weight = 0.5
desc_weight = 1.5
category_weight = 1.0

weighted_features = np.hstack((
    geo_features * geo_weight,
    category_encoded * category_weight
))


In [68]:
# Compute similarity between places
similarity_matrix = cosine_similarity(weighted_features)

In [69]:
# Get recommendations for a specific place (e.g., index 0)
place_index = 123
similar_places = np.argsort(-similarity_matrix[place_index])[1:6]  # Top 5 excluding itself
similar_places


array([2477, 2505, 2840, 2833, 1208])

In [70]:
print(combined_df.iloc[similar_places])

        id                 name  \
2479  3310  Radhakrishna temple   
2507  3381     Rameshwor temple   
2843  3897         Silu Mahadev   
2836  3880  Siddhi Laxmi temple   
1208  1714     Jagannath temple   

                                            description address   latitude  \
2479  \n\nThe Radhakrishna Temple, also known as the...     NaN  27.672074   
2507  \n\nRameshwor Temple, located in Bhaktapur Dur...     NaN  27.672174   
2843  Silu Mahadev is an ancient Shiva temple locate...     NaN  27.672402   
2836  Siddhi Laxmi Temple is a revered Hindu shrine ...     NaN  27.672100   
1208  Heritage Site The ancient evidence says that K...     NaN  27.672100   

      longitude location   types  
2479  85.427638        0  Temple  
2507  85.427471        0  Temple  
2843  85.429017        0  Temple  
2836  85.428100        0  Temple  
1208  85.428100        0  Temple  


In [71]:
indices = pd.Series(combined_df.index, index=combined_df['name'])


In [75]:
def content_recommender(title, cosine_sim=similarity_matrix, indices=indices):
    # Check if the title exists in the indices
    if title not in indices:
        return f"Title '{title}' not found in the dataset."
    
    # Obtain the index of the attraction that matches the title
    idx = indices[title]
    
    # Get the pairwise similarity scores of all attractions with that attraction
    # And convert it into a list of tuples as described above
    sim_scores = list(enumerate(cosine_sim[idx]))
    
    # Sort the attractions based on the cosine similarity scores
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Get the scores of the 10 most similar attractions. Ignore the first one.
    sim_scores = sim_scores[1:11]
    
    # Get the attraction indices
    attraction_indices = [i[0] for i in sim_scores]
    
    # Return the top 10 most similar attractions
    return combined_df.iloc[attraction_indices]

In [76]:
print(combined_df[combined_df["name"]== "Shiva Mandir"])

        id          name                                        description  \
2753  3767  Shiva Mandir  Shiva Mandir, also known as the Sri Shiva Vish...   
5364  6464  Shiva Mandir  \n\nShiva Mandir, also known as Shiv Shani Man...   

                                                address   latitude  longitude  \
2753  M57C+22J, mahaenadara raajamaaraga, kaawaasaot...  27.662582  84.170041   
5364  W33F+8VM Birendra Bazar, ganaeshamaana chaaran...  26.903329  86.074679   

                      location          types  
2753  mahaenadara raajamaaraga   hindu_temple  
5364             ganaeshamaana   hindu_temple  


In [78]:
# Ensure the title exists in the indices
title = "Shiva Mandir"
if title in indices:
	print(content_recommender(title))
else:
	print(f"Title '{title}' not found in the dataset.")

IndexError: index 5364 is out of bounds for axis 0 with size 5364

In [80]:

def visualize_tfidf_tsne(description_vectors):
    # Step 1: Perform t-SNE to reduce the dimensionality of the TF-IDF vectors to 2D
    tsne = TSNE(n_components=2, random_state=0)
    reduced_vectors = tsne.fit_transform(description_vectors)

    # Step 2: Plot the 2D projections of the TF-IDF vectors
    plt.figure(figsize=(10, 8))
    plt.scatter(reduced_vectors[:, 0], reduced_vectors[:, 1])
    plt.title("t-SNE Visualization of TF-IDF Vectorized Descriptions")
    plt.xlabel("t-SNE Component 1")
    plt.ylabel("t-SNE Component 2")
    plt.show()

# Example usage with your pre-existing description_vectors
